# Wareeny — Currency Model Training

Trains YOLOv8m on Egyptian currency dataset and exports to TFLite int8 for the Flutter app.

**Before running:** Runtime → Change runtime type → T4 GPU

**Disconnect-safe:** Training checkpoints save to Google Drive. If Colab disconnects,
re-run all cells — training auto-resumes from where it left off.

**Steps:**
1. Run all cells top to bottom
2. Approve Google Drive access when prompted (Cell 2)
3. Upload `kaggle.json` when prompted (Cell 4)
4. Wait ~60–90 min for training
5. Download `currency_yolov8m.tflite` at the end
6. Paste the printed Dart snippet into `lib/detector.dart`

In [ ]:
# Cell 1 — GPU check
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. Go to Runtime → Change runtime type → T4 GPU.")

gpu = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU : {gpu}")
print(f"VRAM: {vram:.1f} GB")

In [ ]:
# Cell 2 — Mount Google Drive (checkpoints saved here — survives disconnects)
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

DRIVE_DIR = Path("/content/drive/MyDrive/wareeny_training")
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
print(f"Drive folder: {DRIVE_DIR}")

In [ ]:
# Cell 3 — Install dependencies
!pip install -q ultralytics kaggle pillow pyyaml tensorflow

In [ ]:
# Cell 4 — Upload kaggle.json
import os
from google.colab import files

print("Upload your kaggle.json file:")
uploaded = files.upload()

os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
kaggle_path = os.path.expanduser("~/.kaggle/kaggle.json")

with open(kaggle_path, "wb") as f:
    f.write(uploaded["kaggle.json"])

os.chmod(kaggle_path, 0o600)
print("kaggle.json configured.")

In [ ]:
# Cell 5 — Download and prepare dataset
# Always run this after a disconnect — re-downloads dataset (~1 min) and recreates data.yaml.
# Training will still resume from Drive checkpoint in Cell 6.
import os
import shutil
import random
import zipfile
from pathlib import Path
import yaml

DATASET_SLUG = "egyptiris/egyptian-currency"
RAW_DIR      = Path("/content/data/raw")
YOLO_DIR     = Path("/content/data/yolo")
SPLITS       = {"train": 0.8, "val": 0.15, "test": 0.05}
SEED         = 42


def download_dataset():
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    zip_path = RAW_DIR / "dataset.zip"
    if zip_path.exists():
        print("Zip already downloaded, skipping.")
    else:
        print(f"Downloading {DATASET_SLUG} ...")
        os.system(f'kaggle datasets download -d {DATASET_SLUG} -p "{RAW_DIR}"')
        for f in RAW_DIR.glob("*.zip"):
            f.rename(zip_path)
            break
    extract_dir = RAW_DIR / "extracted"
    if not extract_dir.exists():
        print("Extracting ...")
        with zipfile.ZipFile(zip_path, "r") as z:
            z.extractall(extract_dir)
    return extract_dir


def detect_format(extract_dir):
    if any(extract_dir.rglob("labels")):
        return "yolo"
    subdirs = [d for d in extract_dir.iterdir() if d.is_dir()]
    if subdirs:
        return "classification"
    for sub in subdirs:
        if [d for d in sub.iterdir() if d.is_dir()]:
            return "classification"
    return "unknown"


def collect_classification_classes(extract_dir):
    classes = {}
    candidates = [extract_dir]
    top_dirs = [d for d in extract_dir.iterdir() if d.is_dir()]
    if len(top_dirs) == 1:
        candidates = [top_dirs[0]]
    for root in candidates:
        for cls_dir in sorted(root.iterdir()):
            if not cls_dir.is_dir():
                continue
            imgs = [p for p in cls_dir.rglob("*")
                    if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}]
            if imgs:
                classes[cls_dir.name] = imgs
    return classes


def whole_image_bbox_label(class_idx):
    return f"{class_idx} 0.500000 0.500000 1.000000 1.000000\n"


def build_yolo_from_classification(classes):
    class_names = sorted(classes.keys())
    print(f"Classes found ({len(class_names)}): {class_names}")
    random.seed(SEED)
    for split in SPLITS:
        (YOLO_DIR / split / "images").mkdir(parents=True, exist_ok=True)
        (YOLO_DIR / split / "labels").mkdir(parents=True, exist_ok=True)
    for cls_idx, cls_name in enumerate(class_names):
        imgs = classes[cls_name]
        random.shuffle(imgs)
        n       = len(imgs)
        n_train = int(n * SPLITS["train"])
        n_val   = int(n * SPLITS["val"])
        split_imgs = {
            "train": imgs[:n_train],
            "val":   imgs[n_train : n_train + n_val],
            "test":  imgs[n_train + n_val :],
        }
        for split, paths in split_imgs.items():
            for src in paths:
                dst_img = YOLO_DIR / split / "images" / src.name
                dst_lbl = YOLO_DIR / split / "labels" / (src.stem + ".txt")
                if dst_img.exists():
                    dst_img = dst_img.with_stem(f"{cls_name}_{src.stem}")
                    dst_lbl = dst_lbl.with_stem(f"{cls_name}_{src.stem}")
                shutil.copy2(src, dst_img)
                dst_lbl.write_text(whole_image_bbox_label(cls_idx))
        print(f"  {cls_name:30s} — {n} images")
    return class_names


def collect_yolo_classes(extract_dir):
    for yaml_file in extract_dir.rglob("data.yaml"):
        with open(yaml_file) as f:
            cfg = yaml.safe_load(f)
        if "names" in cfg:
            return cfg["names"]
    for txt in extract_dir.rglob("classes.txt"):
        return txt.read_text().splitlines()
    return None


def copy_yolo_dataset(extract_dir):
    images_root = None
    for p in extract_dir.rglob("images"):
        if p.is_dir():
            images_root = p.parent
            break
    if images_root is None:
        raise RuntimeError("Could not find images/ folder inside YOLO dataset.")
    split_dirs = [d for d in (images_root / "images").iterdir() if d.is_dir()]
    if split_dirs and split_dirs[0].name in {"train", "val", "test"}:
        shutil.copytree(images_root, YOLO_DIR, dirs_exist_ok=True)
        print("Copied pre-split YOLO dataset.")
    else:
        all_imgs = [p for p in (images_root / "images").glob("*")
                    if p.suffix.lower() in {".jpg", ".jpeg", ".png"}]
        random.seed(SEED)
        random.shuffle(all_imgs)
        n       = len(all_imgs)
        n_train = int(n * SPLITS["train"])
        n_val   = int(n * SPLITS["val"])
        splits  = {
            "train": all_imgs[:n_train],
            "val":   all_imgs[n_train : n_train + n_val],
            "test":  all_imgs[n_train + n_val :],
        }
        for split, paths in splits.items():
            (YOLO_DIR / split / "images").mkdir(parents=True, exist_ok=True)
            (YOLO_DIR / split / "labels").mkdir(parents=True, exist_ok=True)
            for src in paths:
                shutil.copy2(src, YOLO_DIR / split / "images" / src.name)
                lbl_src = images_root / "labels" / (src.stem + ".txt")
                if lbl_src.exists():
                    shutil.copy2(lbl_src, YOLO_DIR / split / "labels" / lbl_src.name)
        print(f"Split flat YOLO dataset: {n} images.")
    return collect_yolo_classes(extract_dir)


def write_data_yaml(class_names):
    cfg = {
        "path":  str(YOLO_DIR.resolve()),
        "train": "train/images",
        "val":   "val/images",
        "test":  "test/images",
        "nc":    len(class_names),
        "names": class_names,
    }
    out = Path("/content/data/data.yaml")
    with open(out, "w") as f:
        yaml.dump(cfg, f, allow_unicode=True, default_flow_style=False)
    print(f"\ndata.yaml written → {out}")
    print(f"Classes ({len(class_names)}): {class_names}")
    return out


# --- run ---
extract_dir = download_dataset()
fmt = detect_format(extract_dir)
print(f"Dataset format: {fmt}")

if fmt == "classification":
    classes = collect_classification_classes(extract_dir)
    class_names = build_yolo_from_classification(classes)
elif fmt == "yolo":
    class_names = copy_yolo_dataset(extract_dir)
    if class_names is None:
        print("WARNING: could not auto-detect class names. Edit data.yaml manually.")
        class_names = [f"class_{i}" for i in range(20)]
else:
    raise RuntimeError(f"Unknown format. Inspect {extract_dir} manually.")

DATA_YAML = write_data_yaml(class_names)

print("\nDataset stats:")
for split in ["train", "val", "test"]:
    imgs = list((YOLO_DIR / split / "images").glob("*"))
    print(f"  {split:6s}: {len(imgs)} images")

In [ ]:
# Cell 6 — Train YOLOv8m (auto-resumes from Drive checkpoint if disconnected)
import torch
from pathlib import Path
from ultralytics import YOLO

DATA_YAML    = Path("/content/data/data.yaml")
MODEL        = "yolov8m.pt"
EPOCHS       = 150
IMG_SIZE     = 640
DRIVE_DIR    = Path("/content/drive/MyDrive/wareeny_training")
PROJECT      = str(DRIVE_DIR / "runs/currency")
RUN_NAME     = "train"
LAST_WEIGHTS = DRIVE_DIR / "runs/currency/train/weights/last.pt"

vram = torch.cuda.get_device_properties(0).total_memory / 1e9
BATCH = 8 if vram < 6 else 16
print(f"VRAM: {vram:.1f} GB → batch={BATCH}")

if LAST_WEIGHTS.exists():
    # Resume from Drive checkpoint — all hyperparams loaded from saved args.yaml
    print(f"Resuming from {LAST_WEIGHTS} ...")
    model = YOLO(str(LAST_WEIGHTS))
    results = model.train(resume=True)
else:
    # Fresh training run
    print("Starting fresh training ...")
    model = YOLO(MODEL)
    results = model.train(
        data      = str(DATA_YAML),
        epochs    = EPOCHS,
        imgsz     = IMG_SIZE,
        batch     = BATCH,
        device    = 0,
        project   = PROJECT,
        name      = RUN_NAME,
        exist_ok  = True,
        hsv_h     = 0.02,
        hsv_s     = 0.5,
        hsv_v     = 0.4,
        degrees   = 15,
        translate = 0.1,
        scale     = 0.4,
        shear     = 5.0,
        perspective = 0.0005,
        flipud    = 0.0,
        fliplr    = 0.5,
        mosaic    = 1.0,
        mixup     = 0.1,
        optimizer = "AdamW",
        lr0       = 0.001,
        lrf       = 0.01,
        warmup_epochs = 5,
        patience  = 30,
        save_period = 10,
        val       = True,
        plots     = True,
    )

best_weights = DRIVE_DIR / "runs/currency/train/weights/best.pt"
map50 = results.results_dict.get("metrics/mAP50(B)", "N/A")
print(f"\nTraining complete.")
print(f"Best weights: {best_weights}")
print(f"mAP50: {map50}")

In [ ]:
# Cell 7 — Export to TFLite
# Frees GPU/RAM from training before loading TensorFlow — prevents OOM crash.
# If int8 still crashes, set USE_INT8 = False for fp16 (~24 MB, still works in app).
import gc
import shutil
import yaml
import torch
from pathlib import Path
from ultralytics import YOLO

# --- free training memory ---
try:
    del model
except NameError:
    pass
torch.cuda.empty_cache()
gc.collect()

DRIVE_DIR = Path("/content/drive/MyDrive/wareeny_training")
WEIGHTS   = DRIVE_DIR / "runs/currency/train/weights/best.pt"
DATA_YAML = Path("/content/data/data.yaml")
OUTPUT    = Path("/content/currency_yolov8m.tflite")

USE_INT8 = True  # set False if this cell crashes with OOM

if not WEIGHTS.exists():
    raise FileNotFoundError(f"{WEIGHTS} not found. Run Cell 6 first.")

print(f"Exporting → TFLite {'int8' if USE_INT8 else 'fp16'} ...")
model = YOLO(str(WEIGHTS))

model.export(
    format = "tflite",
    imgsz  = 640,
    int8   = USE_INT8,
    data   = str(DATA_YAML) if USE_INT8 else None,
    nms    = False,
)

tflite_candidates = list(WEIGHTS.parent.glob("*.tflite"))
if not tflite_candidates:
    tflite_candidates = list(WEIGHTS.parent.glob("**/*.tflite"))
if not tflite_candidates:
    raise FileNotFoundError("TFLite file not found after export.")

tflite_src = tflite_candidates[0]
shutil.copy2(tflite_src, OUTPUT)
print(f"Saved → {OUTPUT}  ({OUTPUT.stat().st_size / 1e6:.1f} MB)")

# Print Dart snippet
KNOWN_ARABIC = {
    "1": "جنيه واحد",   "1egp": "جنيه واحد",   "1_egp": "جنيه واحد",
    "5": "خمسة جنيه",   "5egp": "خمسة جنيه",   "5_egp": "خمسة جنيه",
    "10": "عشرة جنيه",  "10egp": "عشرة جنيه",  "10_egp": "عشرة جنيه",
    "20": "عشرون جنيه", "20egp": "عشرون جنيه", "20_egp": "عشرون جنيه",
    "25": "خمسة وعشرون جنيه",
    "50": "خمسون جنيه",  "50egp": "خمسون جنيه",  "50_egp": "خمسون جنيه",
    "100": "مية جنيه",  "100egp": "مية جنيه",  "100_egp": "مية جنيه",
    "200": "ميتين جنيه", "200egp": "ميتين جنيه", "200_egp": "ميتين جنيه",
}

with open(DATA_YAML) as f:
    cfg = yaml.safe_load(f)
names = cfg.get("names", [])

print("\n" + "=" * 60)
print("Paste into lib/detector.dart — replace _currencyArabicNames:")
print("=" * 60)
print("final Map<String, String> _currencyArabicNames = {")
for i, name in enumerate(names):
    arabic = KNOWN_ARABIC.get(name.lower().strip(), f"'{name}'  // TODO: add Arabic")
    print(f"  '{i}': '{arabic}',  // {name}")
print("};")
print("=" * 60)

In [ ]:
# Cell 7 — Export to TFLite fp16
# int8 requires TensorFlow calibration which OOMs on free Colab — use fp16 instead.
# fp16 = same accuracy, ~24 MB (vs 12 MB int8), works fine with flutter_vision.
import gc
import shutil
import yaml
import torch
from pathlib import Path
from ultralytics import YOLO

DRIVE_DIR = Path("/content/drive/MyDrive/wareeny_training")
WEIGHTS   = DRIVE_DIR / "runs/currency/train/weights/best.pt"
DATA_YAML = Path("/content/data/data.yaml")
OUTPUT    = Path("/content/currency_yolov8m.tflite")

if not WEIGHTS.exists():
    raise FileNotFoundError(f"{WEIGHTS} not found. Run Cell 6 first.")

print(f"Exporting {WEIGHTS} → TFLite fp16 ...")
model = YOLO(str(WEIGHTS))

model.export(
    format = "tflite",
    imgsz  = 640,
    int8   = False,
    nms    = False,
)

tflite_candidates = list(WEIGHTS.parent.glob("*.tflite"))
if not tflite_candidates:
    tflite_candidates = list(WEIGHTS.parent.glob("**/*.tflite"))
if not tflite_candidates:
    raise FileNotFoundError("TFLite file not found after export.")

tflite_src = tflite_candidates[0]
shutil.copy2(tflite_src, OUTPUT)
print(f"Saved → {OUTPUT}  ({OUTPUT.stat().st_size / 1e6:.1f} MB)")

# Print Dart snippet
KNOWN_ARABIC = {
    "1": "جنيه واحد",   "1egp": "جنيه واحد",   "1_egp": "جنيه واحد",
    "5": "خمسة جنيه",   "5egp": "خمسة جنيه",   "5_egp": "خمسة جنيه",
    "10": "عشرة جنيه",  "10egp": "عشرة جنيه",  "10_egp": "عشرة جنيه",
    "20": "عشرون جنيه", "20egp": "عشرون جنيه", "20_egp": "عشرون جنيه",
    "25": "خمسة وعشرون جنيه",
    "50": "خمسون جنيه",  "50egp": "خمسون جنيه",  "50_egp": "خمسون جنيه",
    "100": "مية جنيه",  "100egp": "مية جنيه",  "100_egp": "مية جنيه",
    "200": "ميتين جنيه", "200egp": "ميتين جنيه", "200_egp": "ميتين جنيه",
}

with open(DATA_YAML) as f:
    cfg = yaml.safe_load(f)
names = cfg.get("names", [])

print("\n" + "=" * 60)
print("Paste into lib/detector.dart — replace _currencyArabicNames:")
print("=" * 60)
print("final Map<String, String> _currencyArabicNames = {")
for i, name in enumerate(names):
    arabic = KNOWN_ARABIC.get(name.lower().strip(), f"'{name}'  // TODO: add Arabic")
    print(f"  '{i}': '{arabic}',  // {name}")
print("};")
print("=" * 60)